<a href="https://colab.research.google.com/github/omsoni/llm-rag-work/blob/rag_deployment/Medical_Assistant_Deployment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Address Colab - Github Compatibility for nbformat

In [ ]:
import json

with open('Medical_Assistant_Deployment.ipynb', 'r', encoding='utf-8') as f:
    nb = json.load(f)

if 'widgets' in nb.get('metadata', {}):
    for widget_key in nb['metadata']['widgets']:
        if 'state' not in nb['metadata']['widgets'][widget_key]:
            nb['metadata']['widgets'][widget_key]['state'] = {}

with open('Medical_Assistant_Deployment.ipynb', 'w', encoding='utf-8') as f:
    json.dump(nb, f, indent=1)

# **Problem Statement**

## Business Context

A sales forecast is a prediction of future sales revenue based on historical data, industry trends, and the status of the current sales pipeline. Businesses use the sales forecast to estimate weekly, monthly, quarterly, and annual sales totals. A company needs to make an accurate sales forecast as it adds value across an organization and helps the different verticals to chalk out their future course of action.

Forecasting helps an organization plan its sales operations by region and provides valuable insights to the supply chain team regarding the procurement of goods and materials. An accurate sales forecast process has many benefits which include improved decision-making about the future and reduction of sales pipeline and forecast risks. Moreover, it helps to reduce the time spent in planning territory coverage and establish benchmarks that can be used to assess trends in the future.

## Objective

Objective is Serialize the model, and expose it as an API. Create a basic User Interface, Dockerize and Deploy the entire stack to Huggingface Space.

## Data Description

The Merck Manuals are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

# **Installing and Importing the necessary libraries**

In [ ]:
#Installing the libraries with the specified versions
!pip install numpy==2.0.2 pandas==2.2.2 scikit-learn==1.6.1 matplotlib==3.10.0 seaborn==0.13.2 joblib==1.4.2 xgboost==2.1.4 requests==2.32.4 huggingface_hub==0.34.0 pipreqs -q

**Note:**

- After running the above cell, kindly restart the notebook kernel (for Jupyter Notebook) or runtime (for Google Colab) and run all cells sequentially from the next cell.

- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in this notebook.

In [1]:
import warnings
warnings.filterwarnings("ignore")

# Libraries to help with reading and manipulating data
import numpy as np
import pandas as pd

# For splitting the dataset
from sklearn.model_selection import train_test_split

# Libaries to help with data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Removes the limit for the number of displayed columns
pd.set_option("display.max_columns", None)
# Sets the limit for the number of displayed rows
pd.set_option("display.max_rows", 100)

# To serialize the model
import joblib

# os related functionalities
import os

# API request
import requests

# for hugging face space authentication to upload files
from huggingface_hub import login, HfApi

# **Loading the dataset**

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os
os.chdir("/content/drive/My Drive/Colab Notebooks/Model Deployment/Medical Assistant")
os.makedirs("deployment_files", exist_ok=True)

# **Deployment - Backend**

## Flask API

The API has two endpoints on for **health check** and one for **posting data** from Streamlit user interface for creating predictions


In [11]:
%%writefile deployment_files/api.py
# api/main.py
from flask import Flask, request, jsonify
from llama_cpp import Llama
from sentence_transformers import CrossEncoder
import chromadb

app = Flask(__name__)

# Load once at startup
llm = Llama(
    model_path="/app/models/llm/Meta-Llama-3-8B-Instruct-Q4_K_M.gguf",
    n_ctx=4096,
    n_gpu_layers=-1,  # -1 = all layers on GPU; 0 = CPU only
    verbose=False
)

reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-12-v2')
chroma_client = chromadb.PersistentClient(path="/app/chroma_db")
collection = chroma_client.get_collection("medical_docs")

# Map your strategy table
SAMPLING_PROFILES = {
    0.0: {"top_p": 0.95, "top_k": 10, "max_tokens": 256},
    0.1: {"top_p": 0.9,  "top_k": 20, "max_tokens": 256},
    0.3: {"top_p": 0.85, "top_k": 40, "max_tokens": 512},
    0.7: {"top_p": 0.9,  "top_k": 50, "max_tokens": 512},
    1.0: {"top_p": 0.95, "top_k": 100,"max_tokens": 1024},
}

def get_sampling_params(temp: float):
    # Snap slider value to nearest profile
    nearest = min(SAMPLING_PROFILES.keys(), key=lambda x: abs(x - temp))
    return SAMPLING_PROFILES[nearest]

@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "ok", "model_loaded": model is not None}), 200

@app.route("/query", methods=["POST"])
def query():
    data = request.get_json(force=True, silent=True)
    if data is None:
        return jsonify({"error": "Request body must be valid JSON"}), 400

    observations = data if isinstance(data, list) else [data]
    predictions  = []

    valid, err = validate_input(data)
    if not valid:
      return jsonify({"error": f"Observation {i}: {err}"}), 422

    query_text = data["query"]
    temperature = float(data.get("temperature", 0.0))

    # 1. Retrieve from Chroma
    results = collection.query(query_texts=[query_text], n_results=20)
    docs = results["documents"][0]

    # 2. Rerank with cross-encoder
    pairs = [[query_text, doc] for doc in docs]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)
    top_docs = [doc for doc, _ in ranked[:5]]

    # 3. Build prompt with context
    context = "\n\n".join(top_docs)
    prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a medical assistant. Answer using only the provided context.

Context:
{context}<|eot_id|><|start_header_id|>user<|end_header_id|>

{query_text}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"""

    # 4. Generate with strategy-based sampling
    params = get_sampling_params(temperature)
    output = llm(
        prompt,
        temperature=temperature,
        top_p=params["top_p"],
        top_k=params["top_k"],
        max_tokens=params["max_tokens"],
        stop=["<|eot_id|>"]
    )

    return jsonify({
        "answer": output["choices"][0]["text"].strip(),
        "strategy": params,
        "sources": top_docs
    })

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)

Writing deployment_files/api.py


## Setting up a Hugging Face Docker Space for the Backend

I created the following public space https://huggingface.co/spaces/omsoni/retail_chain_sales_forecast to deploy user interface and Flask API

# **Deployment - Frontend**

## Streamlit for Interactive UI

In [6]:
%%writefile deployment_files/streamlit_app.py

# streamlit_app.py
import streamlit as st
import requests

# ---- Config ----
API_URL = "http://localhost:5000/query"  # Change to your Flask API URL

STRATEGY_PROFILES = {
    0.0: {"name": "Deterministic", "top_p": 0.95, "top_k": 10,  "max_tokens": 256,  "use_case": "Factual, consistent answers"},
    0.1: {"name": "Conservative",  "top_p": 0.9,  "top_k": 20,  "max_tokens": 256,  "use_case": "Slight variation, still safe"},
    0.3: {"name": "Balanced",      "top_p": 0.85, "top_k": 40,  "max_tokens": 512,  "use_case": "General medical Q&A"},
    0.7: {"name": "Creative",      "top_p": 0.9,  "top_k": 50,  "max_tokens": 512,  "use_case": "Differential diagnosis"},
    1.0: {"name": "Exploratory",   "top_p": 0.95, "top_k": 100, "max_tokens": 1024, "use_case": "Brainstorming, research"},
}

TOP_P_OPTIONS = [0.85, 0.9, 0.95]
TOP_K_OPTIONS = [10, 20, 40, 50, 100]
MAX_TOKENS_OPTIONS = [256, 512, 1024]


def get_nearest_profile(temp: float):
    """Snap temperature to nearest strategy profile."""
    nearest = min(STRATEGY_PROFILES.keys(), key=lambda x: abs(x - temp))
    return nearest, STRATEGY_PROFILES[nearest]


# ---- Page setup ----
st.set_page_config(page_title="Medical RAG Assistant", page_icon="🩺", layout="wide")
st.title("🩺 Medical RAG Assistant")
st.caption("Ask medical questions; answers are grounded in retrieved context.")


# ---- Sidebar: Sampling controls ----
with st.sidebar:
    st.header("⚙️ Generation Settings")

    temperature = st.slider(
        "Temperature",
        min_value=0.0,
        max_value=1.0,
        value=0.0,
        step=0.05,
        help="0 = deterministic, 1 = exploratory"
    )

    # Show which strategy this temperature maps to
    nearest_temp, profile = get_nearest_profile(temperature)
    st.info(f"**Strategy: {profile['name']}**\n\n{profile['use_case']}")

    st.divider()

    # Allow user to override the auto-selected values
    st.subheader("Advanced (override defaults)")

    use_custom = st.checkbox("Customize top_p / top_k / max_tokens", value=False)

    if use_custom:
        top_p = st.selectbox(
            "top_p",
            options=TOP_P_OPTIONS,
            index=TOP_P_OPTIONS.index(profile["top_p"]),
            help="Nucleus sampling threshold"
        )
        top_k = st.selectbox(
            "top_k",
            options=TOP_K_OPTIONS,
            index=TOP_K_OPTIONS.index(profile["top_k"]),
            help="Sample from top K tokens"
        )
        max_tokens = st.selectbox(
            "max_tokens",
            options=MAX_TOKENS_OPTIONS,
            index=MAX_TOKENS_OPTIONS.index(profile["max_tokens"]),
            help="Maximum response length"
        )
    else:
        top_p = profile["top_p"]
        top_k = profile["top_k"]
        max_tokens = profile["max_tokens"]
        st.text(f"top_p:      {top_p}")
        st.text(f"top_k:      {top_k}")
        st.text(f"max_tokens: {max_tokens}")


# ---- Main: Query input ----
query = st.text_area(
    "Your question",
    placeholder="e.g., What are the early symptoms of Type 2 diabetes?",
    height=100
)

col1, col2 = st.columns([1, 5])
with col1:
    submit = st.button("Ask", type="primary", use_container_width=True)
with col2:
    if st.button("Clear", use_container_width=False):
        st.rerun()


# ---- Submit handling ----
if submit and query.strip():
    payload = {
        "query": query,
        "temperature": temperature,
        "top_p": top_p,
        "top_k": top_k,
        "max_tokens": max_tokens,
    }

    with st.spinner("Retrieving context and generating answer..."):
        try:
            response = requests.post(API_URL, json=payload, timeout=120)
            response.raise_for_status()
            data = response.json()
        except requests.exceptions.RequestException as e:
            st.error(f"API request failed: {e}")
            st.stop()

    # ---- Display answer ----
    st.subheader("Answer")
    st.write(data.get("answer", "(no answer returned)"))

    # ---- Display sources ----
    sources = data.get("sources", [])
    if sources:
        with st.expander(f"📚 Retrieved Context ({len(sources)} sources)"):
            for i, src in enumerate(sources, 1):
                st.markdown(f"**Source {i}**")
                st.write(src)
                st.divider()

    # ---- Display settings used ----
    with st.expander("🔧 Generation parameters used"):
        st.json({
            "temperature": temperature,
            "top_p": top_p,
            "top_k": top_k,
            "max_tokens": max_tokens,
            "strategy": profile["name"],
        })

elif submit and not query.strip():
    st.warning("Please enter a question.")

Overwriting deployment_files/streamlit_app.py


## Dependencies File

### Generate Requirements.txt from actual used Python packages

In [7]:
%%writefile deployment_files/requirements.txt

flask==3.0.0
llama-cpp-python==0.2.90
sentence-transformers==3.0.1
chromadb==0.5.5
huggingface_hub==0.24.5
streamlit==1.39.0
requests==2.32.3
gunicorn>=21.2

Writing deployment_files/requirements.txt


## DockerFile

In [8]:
%%writefile deployment_files/Dockerfile

# HuggingFace Spaces runs containers as a non-root user (uid=1000)
# so we must grant ownership of /app explicitly
FROM python:3.11-slim

# System deps for llama-cpp-python
RUN apt-get update && apt-get install -y --no-install-recommends \
    build-essential \
    cmake \
    git \
    curl \
    && rm -rf /var/lib/apt/lists/*

# HF Spaces runs as user 1000 — set this up early
RUN useradd -m -u 1000 user
USER user
ENV PATH="/home/user/.local/bin:$PATH"
WORKDIR /home/user/app

# Python deps
COPY --chown=user requirements.txt .
RUN pip install --no-cache-dir --user -r requirements.txt

# Pre-download models at build time
# Note: bartowski's Llama 3 GGUF is NOT gated, so no HF_TOKEN needed for this one
RUN python -c "from huggingface_hub import hf_hub_download; \
    hf_hub_download(repo_id='bartowski/Meta-Llama-3-8B-Instruct-GGUF', \
    filename='Meta-Llama-3-8B-Instruct-Q4_K_M.gguf', \
    local_dir='/home/user/app/models/llm')"

RUN python -c "from sentence_transformers import CrossEncoder; \
    CrossEncoder('cross-encoder/ms-marco-MiniLM-L-12-v2', \
    cache_folder='/home/user/app/models/reranker')"

# Copy app code last (preserves model layer cache)
COPY --chown=user api.py .
COPY --chown=user streamlit_app.py .
COPY --chown=user start.sh .
RUN chmod +x start.sh

# HF Spaces expects port 7860
EXPOSE 7860

CMD ["./start.sh"]

Writing deployment_files/Dockerfile


In [9]:
%%writefile deployment_files/start.sh

#!/bin/bash
set -e

# Start Flask API in background on port 5000
echo "Starting Flask API..."
python api.py &
FLASK_PID=$!

# Wait for Flask to be ready (max 60s)
echo "Waiting for Flask to be ready..."
for i in {1..30}; do
    if curl -s http://localhost:5000/health > /dev/null 2>&1; then
        echo "Flask is ready!"
        break
    fi
    sleep 1
done

# Trap SIGTERM so both processes shut down cleanly
trap "kill $FLASK_PID; exit" SIGTERM SIGINT

# Start Streamlit in foreground on port 7860 (HF Spaces default)
echo "Starting Streamlit..."
streamlit run streamlit_app.py \
    --server.address=0.0.0.0 \
    --server.port=7860 \
    --server.headless=true \
    --browser.gatherUsageStats=false

Writing deployment_files/start.sh


In [12]:
!ls "./deployment_files"

api.py	Dockerfile  requirements.txt  start.sh	streamlit_app.py


## Uploading Files to Hugging Face Space (Streamlit Space)

In [ ]:
from google.colab import userdata
access_key = userdata.get("HF_TOKEN") ## Hugging Face token created from access keys in write mode
repo_id = "omsoni/medical_assistant"  # Your Hugging Face space id

# Login to Hugging Face platform with the access token
login(token=access_key)

# Initialize the API
api = HfApi()

# Upload Streamlit app files stored in the folder called deployment_files
api.upload_folder(
    folder_path="/content/drive/My Drive/Colab Notebooks/Model Deployment/Medical Assitant/deployment_files",  # Local folder path in azureml
    repo_id=repo_id,  # Hugging face space id
    repo_type="space",  # Hugging face repo type "space"
)

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

  ...il_chain_forecast_model_v1_0.joblib: 100%|##########| 3.83MB / 3.83MB            

CommitInfo(commit_url='https://huggingface.co/spaces/omsoni/retail_chain_sales_forecast/commit/76e3ffdb079ba6ceb170d872ed430d9629927e9b', commit_message='Upload folder using huggingface_hub', commit_description='', oid='76e3ffdb079ba6ceb170d872ed430d9629927e9b', pr_url=None, repo_url=RepoUrl('https://huggingface.co/spaces/omsoni/retail_chain_sales_forecast', endpoint='https://huggingface.co', repo_type='space', repo_id='omsoni/retail_chain_sales_forecast'), pr_revision=None, pr_num=None)

### UI Integrated with Deployed model can be access at space endpoint below:

[Huggingface Space endpoint for Dockerized Forecasting Streamlit App and Final Model](https://huggingface.co/spaces/omsoni/retail_chain_sales_forecast)